In [ ]:
from store import VectorStore

store = VectorStore()
col = store.collection
print(col.count())  # expect 463

In [ ]:
def show(ids):
    r = col.get(ids=ids, include=["documents", "metadatas"])
    for i, d, m in zip(r["ids"], r["documents"], r["metadatas"]):
        print("=" * 80)
        print(i, "\n", m)
        print("-" * 40)
        print(d[:500])

# a normal article, a split article, an annex, a schedule
show(["art-19", "art-175A-p1", "annex-objectives-resolution-p1", "first-schedule-p1"])

# an omitted article (filter, since I don't know its exact ID)
r = col.get(where={"article_number": "184"}, include=["documents", "metadatas"])
print(r["ids"], r["metadatas"], r["documents"][0][:300])

In [ ]:
PREFIX = "Represent this sentence for searching relevant passages: "

def run(q, n=10, prefix=False):
    res = store.query((PREFIX + q) if prefix else q, n_results=n)
    print(f"\nQ: {q}  [prefix={prefix}]")
    for id_, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
        label = meta.get("article_number") or meta.get("section") or meta.get("kind")
        print(f"  {dist:.3f}  {id_:35s} {label}")

for q in ["freedom of speech and the press", "who appoints the Chief Justice"]:
    run(q, prefix=False)
    run(q, prefix=True)

In [ ]:
import re
from dataclasses import dataclass

PREFIX = "Represent this sentence for searching relevant passages: "
INC = ["documents", "metadatas"]
NUM = r"\d{1,3}[A-Za-z]{0,2}"
ART_RE = re.compile(
    rf"\b(?:articles?|arts?\.?)\s+({NUM}(?:\s*(?:,|and|&|to|through|-|–)\s*{NUM})*)", re.I)
MAX_RANGE = 20  # ignore silly ranges like "articles 1 to 280"

def parse_articles(q):
    """-> (list of article_number strings, list of (lo, hi) int ranges)"""
    nums, ranges = [], []
    for m in ART_RE.finditer(q):
        span = m.group(1)
        toks = [t.upper() for t in re.findall(NUM, span)]
        is_range = re.search(r"\bto\b|\bthrough\b|[-–]", span, re.I) and len(toks) == 2
        if is_range:
            lo, hi = (int(re.match(r"\d+", t).group()) for t in toks)
            if 0 <= hi - lo <= MAX_RANGE:
                ranges.append((lo, hi))
                continue
        nums += [t for t in toks if t not in nums]
    return nums, ranges

@dataclass
class Hit:
    key: str            # "19", "175A", "Preamble", "First Schedule", ...
    kind: str
    title: str
    text: str           # all parts, in order
    printed_pages: list
    pdf_pages: list
    amendment_notes: str  # page-level: may include neighbouring articles
    omitted: bool
    distance: float | None
    source: str         # "exact" | "semantic"

def _key(meta):
    return meta.get("article_number") or meta.get("section") or meta.get("kind")

def _build_hit(key, got, distance, source):
    rows = sorted(zip(got["documents"], got["metadatas"]),
                  key=lambda r: r[1].get("chunk_index", 0))
    metas = [m for _, m in rows]
    notes = []
    for m in metas:
        n = m.get("amendment_notes")
        if n and n not in notes:
            notes.append(n)
    first = metas[0]
    return Hit(
        key=key, kind=first.get("kind"),
        title=first.get("article_title") or first.get("section") or first.get("kind"),
        text="\n\n".join(d for d, _ in rows),
        printed_pages=sorted({m["printed_page"] for m in metas if m.get("printed_page")}),
        pdf_pages=sorted({m["pdf_page"] for m in metas if "pdf_page" in m}),
        amendment_notes=" || ".join(notes),
        omitted=any(m.get("omitted") for m in metas),
        distance=distance, source=source)

def _fetch_group(meta):
    where = ({"article_number": meta["article_number"]} if "article_number" in meta
             else {"section": meta["section"]})
    return col.get(where=where, include=INC)

def exact_hits(nums, ranges):
    hits = []
    for n in nums:
        got = col.get(where={"article_number": n}, include=INC)
        if got["ids"]:
            hits.append(_build_hit(n, got, None, "exact"))
    for lo, hi in ranges:
        got = col.get(where={"$and": [{"article_num": {"$gte": lo}},
                                      {"article_num": {"$lte": hi}}]}, include=INC)
        buckets = {}
        for d, m in zip(got["documents"], got["metadatas"]):
            buckets.setdefault(m["article_number"], {"documents": [], "metadatas": []})
            buckets[m["article_number"]]["documents"].append(d)
            buckets[m["article_number"]]["metadatas"].append(m)
        for k in sorted(buckets, key=lambda k: (buckets[k]["metadatas"][0]["article_num"], k)):
            hits.append(_build_hit(k, buckets[k], None, "exact"))
    return hits

def rank_keys(q, n_fetch=30, prefix=True, where=None):
    """Distinct articles/sections ranked by best chunk distance (no sibling fetch, cheap for eval)."""
    res = store.query((PREFIX + q) if prefix else q, n_results=n_fetch, where=where)
    best = {}
    for meta, d in zip(res["metadatas"][0], res["distances"][0]):
        best.setdefault(_key(meta), (d, meta))  # results arrive sorted, first = best
    return [(k, d, m) for k, (d, m) in best.items()]

def search(q, n_fetch=30, n_articles=5, prefix=True):
    nums, ranges = parse_articles(q)
    hits = exact_hits(nums, ranges)
    seen = {h.key for h in hits}
    sem = 0
    for k, d, meta in rank_keys(q, n_fetch, prefix):
        if k in seen:
            continue
        hits.append(_build_hit(k, _fetch_group(meta), d, "semantic"))
        seen.add(k)
        sem += 1
        if sem == n_articles:
            break
    return hits

# quick look
for h in search("How does Article 19 relate to defamation?"):
    print(h.source, h.key, h.title[:50], h.printed_pages, h.omitted, len(h.text))
print(parse_articles("Compare Articles 10 to 12 and Art. 175A"))

In [ ]:
GOLD = [
    ("freedom of speech and the press",                 {"19"}),
    ("minimum age to become President",                 {"41"}),
    ("who appoints the Chief Justice",                  {"175A", "177"}),
    ("can someone be detained without trial",           {"10"}),
    ("right to a fair trial",                           {"10A"}),
    ("right to information",                            {"19A"}),
    ("right to free education for children",            {"25A"}),
    ("are all citizens equal before the law",           {"25"}),
    ("protection of property rights",                   {"24"}),
    ("freedom to practise religion",                    {"20"}),
    ("qualifications to be a member of parliament",     {"62"}),
    ("disqualification of members of parliament",       {"63", "63A"}),
    ("how can the President be impeached",              {"47"}),
    ("proclamation of emergency because of war",        {"232"}),
    ("how is the Constitution amended",                 {"239"}),
    ("what is the national language",                   {"251"}),
    ("Council of Common Interests",                     {"153"}),
    ("who is the Chief Election Commissioner",          {"213"}),
    ("High Court writ jurisdiction habeas corpus",      {"199"}),
    ("local government system",                         {"140A"}),
    ("what is the Objectives Resolution",               {"Annex (Objectives Resolution)", "2A"}),
]

def evaluate(prefix, n_fetch=30, verbose=False):
    ks = (1, 3, 5)
    hit = {k: 0 for k in ks}; rr = 0.0; misses = []
    for q, gold in GOLD:
        keys = [k for k, _, _ in rank_keys(q, n_fetch, prefix)]
        rank = next((i + 1 for i, k in enumerate(keys) if k in gold), None)
        for k in ks:
            hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
        if not rank or rank > 3:
            misses.append((q, gold, rank, keys[:5]))
    n = len(GOLD)
    print(f"prefix={prefix}  hit@1={hit[1]/n:.2f}  hit@3={hit[3]/n:.2f}  "
          f"hit@5={hit[5]/n:.2f}  MRR={rr/n:.2f}")
    if verbose:
        for m in misses: print("   miss:", m)

evaluate(False, verbose=True)
evaluate(True, verbose=True)

In [ ]:
HARD = [
    ("can the government shut down newspapers",                 {"19"}),
    ("what happens if the PM loses majority support",           {"95"}),   # no-confidence; check
    ("who commands the armed forces",                           {"243"}),
    ("can police arrest me without telling me why",             {"10"}),
    ("is untouchability allowed",                               {"11"}),
    ("can a person be forced to work",                          {"11"}),
    ("how long can parliament sit before dissolution",          {"52"}),   # check
    ("can a judge be removed from office",                      {"209", "209A"}),  # 209 may be altered by 26th/27th Amdt.; verify
    ("who can declare someone a non-Muslim",                    {"260"}),
    ("can the army take over civil administration",             {"245"}),
    ("what are the provinces of Pakistan",                      {"1"}),
    ("rights of minorities",                                    {"36"}),
]

GOLD_ALL = GOLD + HARD

def evaluate2(gold, prefix, n_fetch=30):
    ks = (1, 3, 5); hit = {k: 0 for k in ks}; rr = 0.0
    for q, g in gold:
        keys = [k for k, _, _ in rank_keys(q, n_fetch, prefix)]
        rank = next((i + 1 for i, k in enumerate(keys) if k in g), None)
        for k in ks: hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
        if not rank or rank > 3: print("  miss:", q, g, rank, keys[:5])
    n = len(gold)
    print(f"prefix={prefix} hit@1={hit[1]/n:.2f} hit@3={hit[3]/n:.2f} hit@5={hit[5]/n:.2f} MRR={rr/n:.2f}")

evaluate2(HARD, False)
evaluate2(HARD, True)

In [ ]:
for q in ["How does Article 19 relate to defamation?", "can the government shut down newspapers"]:
    print("\n", q)
    for k, d, _ in rank_keys(q, 30, False)[:8]:
        print(f"  {d:.3f}  {k}")

In [ ]:
HARD = [x for x in HARD if "untouchability" not in x[0]]
GOLD_ALL = GOLD + HARD

NOANSWER = [
    "is untouchability allowed",
    "what is the minimum wage in Pakistan",
    "how do I get a driving licence",
]

def sem_keys(q, n_fetch=30, prefix=True):
    return [k for k, _, _ in rank_keys(q, n_fetch, prefix)]

def evaluate4(gold, keyfn, label):
    ks = (1, 3, 5); hit = {k: 0 for k in ks}; rr = 0.0
    for q, g in gold:
        keys = keyfn(q)
        rank = next((i + 1 for i, k in enumerate(keys) if k in g), None)
        for k in ks: hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
    n = len(gold)
    print(f"{label:10s} n={n} hit@1={hit[1]/n:.2f} hit@3={hit[3]/n:.2f} hit@5={hit[5]/n:.2f} MRR={rr/n:.2f}")

evaluate4(GOLD_ALL, sem_keys, "semantic")

ALL = col.get(include=["documents"])
print(col.count(), "chunks")
bad = [m.group() for d in ALL["documents"] for m in re.finditer(r"\b[A-Za-z]{2,}- [a-z]{2,}\b", d)]
print("page-break splits:", len(bad), bad[:20])
for w in ["pro-ceedings", "educa-tional", "jurisdic-tion"]:
    print(w, sum(w in d for d in ALL["documents"]))

In [ ]:
GOLD50 = [
    ("can parliament make laws that violate fundamental rights",         {"8"}),
    ("is it legal to punish someone for an act that was not a crime when committed", {"12"}),
    ("can a person be tried twice for the same offence",                 {"13"}),
    ("can the police search my house without permission",                {"14"}),
    ("freedom to move and settle anywhere in the country",               {"15"}),
    ("right to hold peaceful public meetings",                           {"16"}),
    ("can citizens form political parties or trade unions",              {"17"}),
    ("right to do business or practise a profession",                    {"18"}),
    ("can people be forced to pay tax for another religion",             {"21"}),
    ("religious freedom in educational institutions",                    {"22"}),
    ("is discrimination in public places allowed",                       {"26"}),
    ("reserved seats for women in the National Assembly",                {"51"}),
    ("how long is the term of the National Assembly",                    {"52"}),
    ("how is the Speaker of the National Assembly elected",              {"53"}),
    ("who can dissolve the National Assembly",                           {"58"}),
    ("how many members are in the Senate",                               {"59"}),
    ("what is the term of office of the President",                      {"44"}),
    ("can the President grant pardons",                                  {"45"}),
    ("can the President issue ordinances when parliament is not sitting", {"89"}),
    ("what does a vote of no confidence against the Prime Minister require", {"95"}),
    ("who appoints the Attorney General",                                {"100"}),
    ("who appoints provincial governors",                                {"101"}),
    ("how are revenues shared between the federation and provinces",     {"160"}),
    ("who audits government accounts",                                   {"168", "169"}),
    ("retirement age of Supreme Court judges",                           {"179"}),          # verify
    ("what is the appellate jurisdiction of the Supreme Court",          {"185"}),
    ("are Supreme Court decisions binding on other courts",              {"189"}),
    ("what is the punishment for contempt of court",                     {"204"}),
    ("laws must conform to Islamic injunctions",                         {"227"}),
    ("what does the Council of Islamic Ideology do",                     {"228", "230"}),
    ("can fundamental rights be suspended during an emergency",          {"233"}),
    ("what happens if a provincial government cannot function",          {"234"}),
    ("financial emergency",                                              {"235"}),
    ("who is the supreme commander of the armed forces",                 {"243"}),          # verify (27th Amdt)
    ("can private armed groups be formed",                               {"256"}),
    ("definition of a Muslim",                                           {"260"}),
    ("what is the oath of the Prime Minister",                           {"91", "Third Schedule"}),  # verify
    ("what is high treason",                                             {"6"}),
    ("is Islam the state religion",                                      {"2"}),
    ("promotion of local government institutions",                       {"32"}),
]

def evaluate5(gold, label):
    ks = (1, 3, 5); hit = {k: 0 for k in ks}; rr = 0.0
    for q, g in gold:
        keys = [k for k, _, _ in rank_keys(q, 30, True)]
        rank = next((i + 1 for i, k in enumerate(keys) if k in g), None)
        for k in ks: hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
        if not rank or rank > 1:
            print(f"  rank={rank}  {q}  gold={g}  top5={keys[:5]}")
    n = len(gold)
    print(f"{label} n={n} hit@1={hit[1]/n:.2f} hit@3={hit[3]/n:.2f} hit@5={hit[5]/n:.2f} MRR={rr/n:.2f}")

evaluate5(GOLD50, "GOLD50")

In [ ]:
EXACT = [
    ("What does Article 184 say?",          ["184"]),
    ("explain Article 175A",                ["175A"]),
    ("compare Articles 10 and 10A",         ["10", "10A"]),
    ("Articles 19 to 20",                   ["19", "19A", "20"]),
    ("what is article 2A",                  ["2A"]),
    ("what does Article 400 say?",          []),          # doesn't exist: expect no exact hits
]
for q, want in EXACT:
    hits = search(q)
    got = [h.key for h in hits if h.source == "exact"]
    print("OK " if got == want else "BAD", q, "| exact:", got, "| want:", want,
          "| omitted:", [h.key for h in hits if h.omitted][:3])

In [ ]:
NOANS = ["is untouchability allowed", "what is the minimum wage in Pakistan",
         "how do I get a driving licence", "what is the capital gains tax rate"]
for q in NOANS:
    r = rank_keys(q, 30, True)[:3]
    print(q, "->", [(k, round(d, 3)) for k, d, _ in r])

import statistics
best = [rank_keys(q, 30, True)[0][1] for q, _ in GOLD50]
print("answerable best-distance: min %.3f median %.3f max %.3f" %
      (min(best), statistics.median(best), max(best)))

In [ ]:
import statistics

def ctx_eval(gold, n_articles):
    ok, sizes, misses = 0, [], []
    for q, g in gold:
        hits = search(q, n_articles=n_articles)
        sizes.append(sum(len(h.text) for h in hits))
        if any(h.key in g for h in hits): ok += 1
        else: misses.append(q)
    print(f"n_articles={n_articles}: recall {ok}/{len(gold)}  "
          f"chars median {int(statistics.median(sizes))} max {max(sizes)}")
    for m in misses: print("   miss:", m)

for n in (4, 5, 6, 8):
    ctx_eval(GOLD50, n)

In [ ]:
for n in ("14", "160", "234"):
    r = col.get(where={"article_number": n}, include=["documents"])
    print("=" * 60, n)
    print(r["documents"][0][:800], "\n")

In [ ]:
def search(self, q, n_fetch=40, n_articles=8, max_stubs=2) -> SearchResponse:
    nums, ranges = parse_articles(q)
    hits, missing = self.exact_hits(nums, ranges)
    seen = {h.key for h in hits}
    real, stubs = [], []
    for k, d, meta in self.rank_keys(q, n_fetch):
        if k in seen:
            continue
        is_stub = bool(meta.get("omitted"))
        bucket, limit = (stubs, max_stubs) if is_stub else (real, n_articles)
        if len(bucket) >= limit:
            if len(real) >= n_articles and len(stubs) >= max_stubs:
                break
            continue
        bucket.append(self._build_hit(k, self._fetch_group(meta), d, "semantic"))
        seen.add(k)
    return SearchResponse(hits + real + stubs, missing)

In [ ]:
def search(self, q, n_fetch=40, n_articles=8, max_stubs=2) -> SearchResponse:
    nums, ranges = parse_articles(q)
    hits, missing = self.exact_hits(nums, ranges)
    seen = {h.key for h in hits}
    real, stubs = [], []
    for k, d, meta in self.rank_keys(q, n_fetch):
        if k in seen:
            continue
        is_stub = bool(meta.get("omitted"))
        bucket, limit = (stubs, max_stubs) if is_stub else (real, n_articles)
        if len(bucket) >= limit:
            if len(real) >= n_articles and len(stubs) >= max_stubs:
                break
            continue
        bucket.append(self._build_hit(k, self._fetch_group(meta), d, "semantic"))
        seen.add(k)
    return SearchResponse(hits + real + stubs, missing)

In [ ]:
import importlib, search as search_mod
importlib.reload(search_mod)        # needed if you edit search.py and re-run
from store import VectorStore
from search import ConstitutionSearcher, format_context, wants_amendments, parse_articles

s = ConstitutionSearcher(VectorStore())

r = s.search("what does Article 184 say?")
print([(h.key, h.source, h.omitted) for h in r.hits])   # 184 exact first; stubs only at the end

r = s.search("how many members are in the Senate")
print(len(r.hits), [h.key for h in r.hits if h.omitted])  # at most 8 real + 2 stubs

r = s.search("what does Article 400 say?")
print(r.missing)                                          # ['400']

r = s.search("what is the Objectives Resolution")
print([h.key for h in r.hits][:3])                        # no KeyError

In [ ]:
r = s.search("what does Article 14 say?")
print(r.hits[0].key, "|", r.hits[0].title, "|", r.hits[0].printed_pages)
print(r.hits[0].text)          # full article, header line included

print(format_context(r)[:1500])  # exactly what the LLM will receive